In [ ]:
import typesense
from dotenv import load_dotenv
from pathlib import Path
import os

load_dotenv()

In [ ]:
api_key = os.getenv('TYPESENSE_API_KEY')
if not api_key:
    raise RuntimeError('TYPESENSE_API_KEY is missing. Add it to the project .env file.')

client = typesense.Client({
  'nodes': [{
    'host': 'p6ztcf5hybnxavmsp-1.a2.typesense.net',  # For Typesense Cloud use xxx.a1.typesense.net
    'port': '443',       # For Typesense Cloud use 443
    'protocol': 'https'    # For Typesense Cloud use https
  }],
  'api_key': api_key,
  'connection_timeout_seconds': 5
})

books_schema = {
  'name': 'books',
  'fields': [
    {'name': 'title', 'type': 'string'},
    {'name': 'authors', 'type': 'string[]', 'facet': True},
    {'name': 'publication_year', 'type': 'int32', 'facet': True},
    {'name': 'ratings_count', 'type': 'int32'},
    {'name': 'average_rating', 'type': 'float'}
  ],
  'default_sorting_field': 'ratings_count'
}

collection_names = {collection['name'] for collection in client.collections.retrieve()}
if books_schema['name'] not in collection_names:
    collection = client.collections.create(books_schema)
else:
    collection = client.collections[books_schema['name']].retrieve()

print(f"Collection ready: {collection['name']}")

In [ ]:
client

In [ ]:
data_path = Path('books.jsonl')
if not data_path.exists():
    data_path = Path('Typesense/books.jsonl')

with data_path.open('r', encoding='utf-8') as jsonl_file:
    data = jsonl_file.read()
    result = client.collections['books'].documents.import_(data, {'action': 'upsert'})

failed = [item for item in result if not item.get('success')]
print(f"Imported {len(result) - len(failed)} documents; {len(failed)} failed.")